In [1]:
from utils.analysis.portfolio import PortfolioAnalyzer, PortfolioConfig
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE

In [2]:
# 📊 CONFIGURACIÓN DEL PORTFOLIO
# Valores por defecto vienen de config.py (PORTFOLIO_CONFIG)
# Personaliza aquí si necesitas valores diferentes:

config = PortfolioConfig(
    min_score=60.0,              # Puntaje mínimo de calidad (default: 60.0)
    max_companies=10,            # Máximo de empresas en portfolio (default: 10)
    max_per_sector=3,            # Máximo por sector (default: 3)
    selection_method='balanced',    # Método: 'balanced', 'value', 'growth', 'total_score'
    weight_method='black_litterman',  # Método: 'equal', 'score', 'score_risk_adjusted', 'markowitz', 'black_litterman'
    lookback_years=3             # Años históricos (default: 5)
)

print("✅ Configuración creada")
print(f"   Min Score: {config.min_score}")
print(f"   Max Companies: {config.max_companies}")
print(f"   Selection: {config.selection_method}")
print(f"   Weights: {config.weight_method}")
print(f"   Lookback: {config.lookback_years} años")

✅ Configuración creada
   Min Score: 60.0
   Max Companies: 10
   Selection: balanced
   Weights: black_litterman
   Lookback: 3 años


In [3]:
# 🔧 INICIALIZACIÓN
# Crear instancia compartida de DataManager para eficiencia
data_manager = DataManager()

# Crear analyzer pasándole la configuración y el data_manager
analyzer = PortfolioAnalyzer(config=config, data_manager=data_manager)

print("\n📅 Fechas de análisis (desde config.py):")
print(f"   Inicio: {ANALYSIS_START_DATE}")
print(f"   Fin: {ANALYSIS_END_DATE}")
print("\n✅ Analyzer inicializado y listo para usar")


📅 Fechas de análisis (desde config.py):
   Inicio: 2020-01-01
   Fin: 2025-12-24

✅ Analyzer inicializado y listo para usar


In [4]:
# 📈 EJEMPLO 1: Analizar componentes de un índice
# Índices disponibles: 'SP500', 'NASDAQ100', 'DOW30'

print("🔍 Analizando S&P 500...")
result = analyzer.analyze_from_index('SP500')

if result.get('success'):
    print(f"\n✅ Portfolio creado exitosamente!")
    print(f"\n📊 Empresas seleccionadas: {len(result['tickers'])}")
    for ticker in result['tickers']:
        weight = result['weights'].get(ticker, 0)
        print(f"   {ticker}: {weight*100:.2f}%")
    
    print(f"\n📈 Métricas del portfolio:")
    metrics = result['metrics']
    for key, value in metrics.items():
        print(f"   {key}: {value}")
    
    print(f"\n📅 Período analizado:")
    print(f"   {result['period']['start']} → {result['period']['end']}")
else:
    print(f"❌ Error: {result.get('error')}")

🔍 Analizando S&P 500...
✅ Obtenidos 503 componentes del S&P 500
⚡ Fase 1: Análisis rápido de todas las 503 empresas para identificar mejores candidatas...
   ⏱️  Esto puede tomar varios minutos debido a rate limiting de Yahoo Finance...
📊 Análisis rápido: 50/503 empresas (50 exitosas)...
📊 Análisis rápido: 100/503 empresas (100 exitosas)...
📊 Análisis rápido: 150/503 empresas (150 exitosas)...
📊 Análisis rápido: 200/503 empresas (200 exitosas)...
📊 Análisis rápido: 250/503 empresas (250 exitosas)...
📊 Análisis rápido: 300/503 empresas (300 exitosas)...
📊 Análisis rápido: 350/503 empresas (350 exitosas)...
📊 Análisis rápido: 400/503 empresas (400 exitosas)...
📊 Análisis rápido: 450/503 empresas (450 exitosas)...
📊 Análisis rápido: 500/503 empresas (500 exitosas)...
📊 Análisis rápido: 503/503 empresas (503 exitosas)...
   ✅ 57 empresas superan el min_score de 60.0
⚡ Fase 2: Análisis completo de 50 mejores candidatas...
📊 Progreso: 10/50 empresas analizadas...
📊 Progreso: 20/50 empresas a

In [5]:
# 📈 EJEMPLO 2: Analizar tickers específicos
# Útil para portfolios personalizados

custom_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'JPM', 'V', 'WMT']

print(f"🔍 Analizando {len(custom_tickers)} empresas personalizadas...")
result = analyzer.analyze(custom_tickers)

if result.get('success'):
    print(f"\n✅ Portfolio creado!")
    print(f"📊 Seleccionadas: {len(result['tickers'])}/{len(custom_tickers)}")
    print(f"\nDistribución de pesos:")
    for ticker in result['tickers']:
        weight = result['weights'].get(ticker, 0)
        score = result['analysis'][ticker]['scores']['total']
        print(f"   {ticker}: {weight*100:5.2f}% (Score: {score:.1f})")
else:
    print(f"❌ Error: {result.get('error')}")

🔍 Analizando 10 empresas personalizadas...
📊 Progreso: 10/10 empresas analizadas...
✅ Se analizaron 10 empresas correctamente
🎯 Seleccionando mejores empresas según criterios...
   - Método: balanced
   - Min score: 60.0
   - Max companies: 10
   📊 Convirtiendo 10 resultados a DataFrame...
   📊 DataFrame tiene 10 filas
   🔍 Filtrando por min_score >= 60.0...
   ✅ 5 empresas superan el min_score
   📈 Aplicando método de scoring: balanced
   🎲 Aplicando diversificación (max_per_sector: 3)...
   ✅ Seleccionadas 5 empresas tras diversificación
✅ Seleccionadas 5 empresas: NVDA, MSFT, META, GOOGL, AAPL
📥 Descargando datos históricos para 5 empresas...
Período: 2020-01-01 → 2025-12-24
📊 Calculando retornos históricos...
⚖️  Optimizando pesos del portfolio (método: black_litterman)...
📈 Calculando métricas del portfolio...
✅ Portfolio construido exitosamente con 5 empresas
📦 Preparando respuesta...

✅ Portfolio creado!
📊 Seleccionadas: 5/10

Distribución de pesos:
   NVDA: 26.09% (Score: 67.5)

In [6]:
# 📊 EJEMPLO 3: Comparar diferentes métodos de selección
# Métodos disponibles: 'balanced', 'value', 'growth', 'total_score'

test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'JPM', 'BAC', 'WMT', 'PG', 'JNJ']
methods = ['balanced', 'value', 'growth', 'total_score']

print("🔬 Comparando métodos de selección:\n")

for method in methods:
    config_test = PortfolioConfig(
        min_score=60.0,
        max_companies=5,
        selection_method=method,
        weight_method='equal'
    )
    
    analyzer_test = PortfolioAnalyzer(config=config_test, data_manager=data_manager)
    result = analyzer_test.analyze(test_tickers)
    
    if result.get('success'):
        print(f"📈 Método '{method}':")
        print(f"   Seleccionados: {', '.join(result['tickers'])}")
    else:
        print(f"❌ Método '{method}': {result.get('error')}")
    print()

🔬 Comparando métodos de selección:

📊 Progreso: 10/10 empresas analizadas...
✅ Se analizaron 10 empresas correctamente
🎯 Seleccionando mejores empresas según criterios...
   - Método: balanced
   - Min score: 60.0
   - Max companies: 5
   📊 Convirtiendo 10 resultados a DataFrame...
   📊 DataFrame tiene 10 filas
   🔍 Filtrando por min_score >= 60.0...
   ✅ 7 empresas superan el min_score
   📈 Aplicando método de scoring: balanced
   🎲 Aplicando diversificación (max_per_sector: 3)...
   ✅ Seleccionadas 5 empresas tras diversificación
✅ Seleccionadas 5 empresas: JNJ, NVDA, MSFT, META, GOOGL
⚖️  Optimizando pesos del portfolio (método: equal)...
📈 Calculando métricas del portfolio...
✅ Portfolio construido exitosamente con 5 empresas
📦 Preparando respuesta...
📈 Método 'balanced':
   Seleccionados: JNJ, NVDA, MSFT, META, GOOGL

📊 Progreso: 10/10 empresas analizadas...
✅ Se analizaron 10 empresas correctamente
🎯 Seleccionando mejores empresas según criterios...
   - Método: value
   - Min sc

In [7]:
# ⚖️ EJEMPLO 4: Comparar métodos de optimización de pesos
# Métodos: 'equal', 'score', 'score_risk_adjusted', 'markowitz', 'black_litterman'

weight_methods = {
    'equal': 'Equal Weight (1/n)',
    'score': 'Score-Based (proporcional a fundamentales)',
    'score_risk_adjusted': 'Score/Risk (score ÷ volatilidad)',
    'markowitz': 'Markowitz (máx Sharpe Ratio)',
    'black_litterman': 'Black-Litterman (equilibrio + views)',
}

test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'JPM', 'BAC', 'WMT', 'PG', 'JNJ']

print("⚖️  Comparando métodos de optimización de pesos:\n")

for method_key, method_name in weight_methods.items():
    config_test = PortfolioConfig(
        min_score=60.0,
        max_companies=8,
        selection_method='balanced',
        weight_method=method_key,
        lookback_years=3
    )
    
    analyzer_test = PortfolioAnalyzer(config=config_test, data_manager=data_manager)
    result = analyzer_test.analyze(test_tickers)
    
    if result.get('success'):
        print(f"📊 {method_name}:")
        # Ordenar por peso descendente
        sorted_weights = sorted(result['weights'].items(), key=lambda x: x[1], reverse=True)
        for ticker, weight in sorted_weights:
            if weight > 0.001:  # Solo mostrar pesos > 0.1%
                print(f"   {ticker}: {weight*100:5.2f}%")
        print(f"   Score ponderado: {result['metrics']['total_score']:.1f}")
    else:
        print(f"❌ {method_name}: {result.get('error')}")
    print()

⚖️  Comparando métodos de optimización de pesos:

📊 Progreso: 10/10 empresas analizadas...
✅ Se analizaron 10 empresas correctamente
🎯 Seleccionando mejores empresas según criterios...
   - Método: balanced
   - Min score: 60.0
   - Max companies: 8
   📊 Convirtiendo 10 resultados a DataFrame...
   📊 DataFrame tiene 10 filas
   🔍 Filtrando por min_score >= 60.0...
   ✅ 7 empresas superan el min_score
   📈 Aplicando método de scoring: balanced
   🎲 Aplicando diversificación (max_per_sector: 3)...
   ✅ Seleccionadas 7 empresas tras diversificación
✅ Seleccionadas 7 empresas: JNJ, NVDA, MSFT, META, GOOGL, AAPL, BAC
⚖️  Optimizando pesos del portfolio (método: equal)...
📈 Calculando métricas del portfolio...
✅ Portfolio construido exitosamente con 7 empresas
📦 Preparando respuesta...
📊 Equal Weight (1/n):
   JNJ: 14.29%
   NVDA: 14.29%
   MSFT: 14.29%
   META: 14.29%
   GOOGL: 14.29%
   AAPL: 14.29%
   BAC: 14.29%
   Score ponderado: 63.6

📊 Progreso: 10/10 empresas analizadas...
✅ Se anal

In [8]:
# 🏆 EJEMPLO 5: Portfolio profesional con Black-Litterman
# BL combina equilibrio de mercado + tus scores fundamentales
# Es el modelo más usado en gestión institucional de activos

print("🏆 Creando portfolio Black-Litterman profesional...\n")

config_bl = PortfolioConfig(
    min_score=65.0,
    max_companies=10,
    max_per_sector=3,
    selection_method='balanced',
    weight_method='black_litterman',
    lookback_years=5
)

analyzer_bl = PortfolioAnalyzer(config=config_bl, data_manager=data_manager)
result = analyzer_bl.analyze_from_index('NASDAQ100')

if result.get('success'):
    print("✅ Portfolio Black-Litterman creado!\n")
    
    print("⚖️  Pesos optimizados:")
    sorted_weights = sorted(result['weights'].items(), key=lambda x: x[1], reverse=True)
    for ticker, weight in sorted_weights:
        if weight > 0.001:
            score = result['analysis'][ticker]['scores']['total']
            print(f"   {ticker}: {weight*100:5.2f}%  (Score: {score:.1f})")
    
    print(f"\n📈 Métricas:")
    print(f"   Score ponderado: {result['metrics']['total_score']:.1f}")
    print(f"   Empresas: {result['metrics']['num_companies']}")
    print(f"\n🏢 Diversificación sectorial:")
    for sector, alloc in sorted(result['metrics']['sector_allocation'].items(), key=lambda x: x[1], reverse=True):
        print(f"   {sector}: {alloc*100:.1f}%")
    
    print(f"\n📅 Período: {result['period']['start']} → {result['period']['end']}")
else:
    print(f"❌ Error: {result.get('error')}")

🏆 Creando portfolio Black-Litterman profesional...

✅ Obtenidos 101 componentes del NASDAQ-100
⚡ Fase 1: Análisis rápido de todas las 101 empresas para identificar mejores candidatas...
   ⏱️  Esto puede tomar varios minutos debido a rate limiting de Yahoo Finance...
📊 Análisis rápido: 50/101 empresas (50 exitosas)...
📊 Análisis rápido: 100/101 empresas (100 exitosas)...
📊 Análisis rápido: 101/101 empresas (101 exitosas)...
   ✅ 5 empresas superan el min_score de 65.0
⚡ Fase 2: Análisis completo de 5 mejores candidatas...
📊 Progreso: 5/5 empresas analizadas...
✅ Se analizaron 5 empresas correctamente
🎯 Seleccionando mejores empresas según criterios...
   - Método: balanced
   - Min score: 65.0
   - Max companies: 10
   📊 Convirtiendo 5 resultados a DataFrame...
   📊 DataFrame tiene 5 filas
   🔍 Filtrando por min_score >= 65.0...
   ✅ 5 empresas superan el min_score
   📈 Aplicando método de scoring: balanced
   🎲 Aplicando diversificación (max_per_sector: 3)...
   ✅ Seleccionadas 5 empr